In [0]:
from pyspark.sql import functions as F

# ============================================================
# Paths
# ============================================================

dataset_path = (
    "dbfs:/Volumes/"
    "ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_oe_journey/"
    "lgbm_feature_selection/"
    "feature_selection_mdb_25k_final_parquet"
)

mdb_path = (
    "dbfs:/Volumes/"
    "ph-com-ai-us-prod-catalog-2553213839599380/"
    "oe_journey/"
    "oe_volume_space/"
    "notebook_output/"
    "model_training_mdb_v6_p12_added"
)

# ============================================================
# Load data in Spark
# ============================================================

dataset = spark.read.parquet(dataset_path)

In [0]:
mdb = spark.read.format("delta").load(mdb_path)

In [0]:
print("DATASET SHAPE:", (dataset.count(), len(dataset.columns)))

In [0]:
print("MDB SHAPE:", (mdb.count(), len(mdb.columns)))

In [0]:
nunique_provider_id = dataset.select("PROVIDER_ID").distinct().count()
print("Number of unique PROVIDER_IDs in dataset:", nunique_provider_id)

In [0]:
nunique_bh_id = mdb.select("CUSTOMER_ID").distinct().count()
print("Number of unique BH_IDs (CUSTOMER_ID) in mdb:", nunique_bh_id)

In [0]:
before_shape = (
    mdb.count(),
    len(mdb.columns)
)

df_dedup = mdb.dropDuplicates(["CUSTOMER_ID",'INSTANCE'])

after_shape = (
    df_dedup.count(),
    len(df_dedup.columns)
)

print("Before shape:", before_shape)
print("After shape :", after_shape)
print("Removed rows:", before_shape[0] - after_shape[0])

In [0]:
dataset.limit(5).display()

In [0]:
mdb.limit(5).display()

In [0]:
mdb.select("INSTANCE").distinct().show(truncate=False)

# If you want the unique values as a Python list
unique_instances = [
    r["INSTANCE"]
    for r in mdb.select("INSTANCE").distinct().orderBy("INSTANCE").collect()
]

print("Unique INSTANCE values:", unique_instances)


In [0]:
provider_bh_mapping = spark.read.format("delta").load(
    "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/oe_journey/oe_volume_space/query_output/kerendia_nba_3_0_npi_bhid_mapping"
).select("PROVIDER_ID", "BH_ID").distinct()

In [0]:
provider_bh_mapping.limit(5).display()

In [0]:
mdb_with_provider_id = mdb.join(provider_bh_mapping, mdb["CUSTOMER_ID"] == provider_bh_mapping["BH_ID"], how="inner")
display(mdb_with_provider_id.select("PROVIDER_ID").distinct())

In [0]:
print("Shape:", (mdb_with_provider_id.count(), len(mdb_with_provider_id.columns)))

In [0]:
print(mdb_with_provider_id.head(2))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# Update PROVIDER_ID by appending INSTANCE, then drop INSTANCE
mdb_with_provider_id = (
    mdb_with_provider_id
    .withColumn(
        "PROVIDER_ID",
        F.concat(
            F.col("PROVIDER_ID").cast(StringType()),
            F.col("INSTANCE").cast(StringType())
        )
    )
    .drop("INSTANCE")
)

In [0]:
nunique_provider_id = mdb_with_provider_id.select("PROVIDER_ID").distinct().count()
print("Number of unique PROVIDER_IDs:", nunique_provider_id)

In [0]:
dataset.limit(2).display()

In [0]:
mdb_with_provider_id.limit(2).display()

In [0]:
shape = (
    mdb_with_provider_id.count(),
    len(mdb_with_provider_id.columns)
)

print("Shape:", shape)

In [0]:
nunique_provider_id = mdb_with_provider_id.select("PROVIDER_ID").distinct().count()
print("Number of unique PROVIDER_IDs:", nunique_provider_id)

In [0]:
from pyspark.sql.types import LongType
from pyspark.sql import functions as F

mdb_with_provider_id = mdb_with_provider_id.withColumn(
    "PROVIDER_ID",
    F.col("PROVIDER_ID").cast(LongType())
)

dataset = dataset.withColumn(
    "PROVIDER_ID",
    F.col("PROVIDER_ID").cast(LongType())
)

In [0]:
nunique_provider_id = mdb_with_provider_id.select("PROVIDER_ID").distinct().count()
print("Number of unique PROVIDER_IDs:", nunique_provider_id)

In [0]:
nunique_provider_id = dataset.select("PROVIDER_ID").distinct().count()
print("Number of unique PROVIDER_IDs:", nunique_provider_id)

In [0]:
# Unique PROVIDER_IDs from both datasets
mdb_ids = mdb_with_provider_id.select("PROVIDER_ID").distinct()

dataset_ids = dataset.select("PROVIDER_ID").distinct()

# Count overlapping PROVIDER_IDs
overlap_count = (
    mdb_ids
    .join(dataset_ids, on="PROVIDER_ID", how="inner")
    .count()
)

print("Overlapping PROVIDER_ID count:", overlap_count)

In [0]:
# Keep only PROVIDER_ID and columns that contain any of the target keywords
# AND contain 'p12' — case insensitive

target_keywords = [
    "general_count", "patient_profile", "samples", "safety_efficacy",
    "basic_product_info", "disease_awareness_management", "study_design",
    "mechanism_of_action", "unmet_need", "access_coverage",
    "dosing_administration", "cost", "guidelines"
]

cols_to_keep = ["PROVIDER_ID"] + [
    c for c in mdb_with_provider_id.columns
    if "p12" in c.lower()
    and any(kw in c.lower() for kw in target_keywords)
]

mdb_with_provider_id = mdb_with_provider_id.select(cols_to_keep)

print(f"Kept {len(cols_to_keep)} columns:")
for c in cols_to_keep:
    print(f"  {c}")

In [0]:
before_shape = (mdb_with_provider_id.count(), len(mdb_with_provider_id.columns))
mdb_with_provider_id_dedup = mdb_with_provider_id.dropDuplicates()
after_shape = (mdb_with_provider_id_dedup.count(), len(mdb_with_provider_id_dedup.columns))
print("Before dropDuplicates:", before_shape)
print("After dropDuplicates :", after_shape)
print("Removed rows:", before_shape[0] - after_shape[0])

In [0]:
dataset.limit(5).display()

In [0]:
mdb_with_provider_id.limit(5).display()

In [0]:
from pyspark.sql.types import FloatType


dataset = dataset.join(mdb_with_provider_id, on="PROVIDER_ID", how="left")

# Columns coming from mdb_with_provider_id (everything except PROVIDER_ID)
mdb_cols = [c for c in mdb_with_provider_id.columns if c != "PROVIDER_ID"]

# Count nulls for each of those columns in the joined dataset
null_counts = dataset.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in mdb_cols
])

null_counts.show(vertical=True)
print(f"Total rows in dataset: {dataset.count()}")

In [0]:
print(f"Rows: {dataset.count()}, Columns: {len(dataset.columns)}")

In [0]:
# Any column containing P12 anywhere in name
p12_cols = [c for c in dataset.columns if "P12" in c.upper()]

print("Total columns:", len(dataset.columns))
print("Columns containing P12:", len(p12_cols))

for c in p12_cols:
    print(c)

In [0]:
# ============================================================
# Save final dataset as Parquet
# ============================================================

PARQUET_PATH_SPARK = (
    "dbfs:/Volumes/"
    "ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_oe_journey/"
    "lgbm_feature_selection/"
    "final_dataset_parquet_v1"
)

PARQUET_PATH_PYTHON = (
    "/Volumes/"
    "ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_oe_journey/"
    "lgbm_feature_selection/"
    "final_dataset_parquet_v1"
)

# Save your final Spark dataframe
dataset.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(PARQUET_PATH_SPARK)

print("Parquet saved successfully.")
print("Spark path :", PARQUET_PATH_SPARK)
print("Python path:", PARQUET_PATH_PYTHON)